In [5]:
import pandas as pd
import numpy as np
import pickle
import os
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import save_npz, load_npz


In [6]:
print("Đang tải dữ liệu bài báo (Có stopwords)...")
df_with_sw = pd.read_pickle('data/news_dataset_df_prep2.pkl')
print(f"-> Tải thành công {len(df_with_sw)} bài báo có stopwords.")

print("Đang tải dữ liệu bài báo (Không có stopwords)...")
df_no_sw = pd.read_pickle('data/news_dataset_df_prep3.pkl')
print(f"-> Tải thành công {len(df_no_sw)} bài báo không có stopwords.")

Đang tải dữ liệu bài báo (Có stopwords)...
-> Tải thành công 160254 bài báo có stopwords.
Đang tải dữ liệu bài báo (Không có stopwords)...
-> Tải thành công 160254 bài báo không có stopwords.


In [7]:
print("Đang tải Inverted Index (Có stopwords)...")
with open("data/inverted_index.pkl", "rb") as f:
    inverted_index_with_sw = pickle.load(f)

print("Đang tạo Inverted Index (Không có stopwords)...")
inverted_index_no_sw = defaultdict(list)
for doc_id, text in zip(df_no_sw['id'], df_no_sw['combined_processed']):
    for token in set(str(text).split()):
        inverted_index_no_sw[token].append(doc_id)
print(f"-> Đã tạo xong Inverted Index không stopwords với {len(inverted_index_no_sw)} từ khóa.")

Đang tải Inverted Index (Có stopwords)...
Đang tạo Inverted Index (Không có stopwords)...
-> Đã tạo xong Inverted Index không stopwords với 518379 từ khóa.


In [8]:
doc_id_to_idx = {doc_id: idx for idx, doc_id in enumerate(df_with_sw['id'])}
print("-> Đã chuẩn bị xong bộ lọc và bộ ánh xạ.")

-> Đã chuẩn bị xong bộ lọc và bộ ánh xạ.


In [9]:
print("Đang xây dựng ma trận TF-IDF cho cả 2 mô hình... (Có thể mất 2-3 phút)")

# --- MÔ HÌNH 1: CÓ STOPWORDS ---
vectorizer_with_sw = TfidfVectorizer(
    min_df=5, max_df=0.8, max_features=100000, lowercase=True
)
tfidf_matrix_with_sw = vectorizer_with_sw.fit_transform(df_with_sw['combined_processed'].astype(str))

# --- MÔ HÌNH 2: KHÔNG CÓ STOPWORDS ---
vectorizer_no_sw = TfidfVectorizer(
    min_df=5, max_df=0.8, max_features=100000, lowercase=True
)
tfidf_matrix_no_sw = vectorizer_no_sw.fit_transform(df_no_sw['combined_processed'].astype(str))

print(f"[Có Stopwords] Kích thước từ vựng: {len(vectorizer_with_sw.vocabulary_)} | Ma trận: {tfidf_matrix_with_sw.shape}")
print(f"[Không Stopwords] Kích thước từ vựng: {len(vectorizer_no_sw.vocabulary_)} | Ma trận: {tfidf_matrix_no_sw.shape}")

Đang xây dựng ma trận TF-IDF cho cả 2 mô hình... (Có thể mất 2-3 phút)
[Có Stopwords] Kích thước từ vựng: 80106 | Ma trận: (160254, 80106)
[Không Stopwords] Kích thước từ vựng: 79428 | Ma trận: (160254, 79428)


In [10]:
def search_documents_inverted(query_text, k, vectorizer, tfidf_matrix, df_documents, inverted_index, doc_id_to_idx, model_name=""):
    print(f"\n{'='*60}")
    print(f"**Truy vấn [{model_name}]:** '{query_text}'")
    
    # LỌC THÔ (Retrieval) dùng Inverted Index
    query_tokens = vectorizer.build_analyzer()(query_text)
    
    candidate_doc_ids = set()
    for token in query_tokens:
        if token in inverted_index:
            candidate_doc_ids.update(inverted_index[token])
            
    print(f"-> Bộ lọc tìm thấy: {len(candidate_doc_ids)} văn bản ứng viên.")
    
    if not candidate_doc_ids:
        print("Không tìm thấy tài liệu nào chứa từ khóa từ truy vấn.")
        return []
        
    candidate_indices = [doc_id_to_idx[d_id] for d_id in candidate_doc_ids if d_id in doc_id_to_idx]
    
    # XẾP HẠNG (Ranking) dùng TF-IDF
    query_vector = vectorizer.transform([query_text])
    sub_tfidf_matrix = tfidf_matrix[candidate_indices]
    
    similarities = cosine_similarity(query_vector, sub_tfidf_matrix).flatten()
    top_k_sub_indices = similarities.argsort()[-k:][::-1]
    
    print(f"--- Top {k} Tài liệu tương đồng nhất ---")
    results = []
    for rank, sub_idx in enumerate(top_k_sub_indices):
        score = similarities[sub_idx]
        if score <= 0:
            continue
            
        original_idx = candidate_indices[sub_idx]
        doc_id = df_documents.iloc[original_idx]['id']
        doc_title = df_documents.iloc[original_idx]['title']
        doc_content = df_documents.iloc[original_idx]['combined_processed']
        
        print(f"\nRank {rank + 1} (Score: {score:.4f}):")
        print(f"  ID: {doc_id} | Title: {doc_title}")
        print(f"  Content: {str(doc_content)[:150]}...")
        
        results.append({
            'rank': rank + 1,
            'similarity_score': score,
            'doc_id': doc_id,
            'title': doc_title
        })
        
    return results

In [11]:
query = "cướp tiệm vàng ở huế"
k_results = 5

# Test với mô hình CÓ Stopwords
results_with_sw = search_documents_inverted(
    query_text=query, 
    k=k_results, 
    vectorizer=vectorizer_with_sw, 
    tfidf_matrix=tfidf_matrix_with_sw, 
    df_documents=df_with_sw, 
    inverted_index=inverted_index_with_sw, 
    doc_id_to_idx=doc_id_to_idx,
    model_name="CÓ Stopwords"
)

# Test với mô hình KHÔNG Stopwords
# (Nếu query có stopword như chữ "ở", bộ tokenizer của vectorizer_no_sw vẫn sẽ tách ra.
# Tốt nhất là sử dụng một câu query đã lọc stopwords nếu muốn tối ưu).
query_clean = "cướp tiệm vàng huế"

results_no_sw = search_documents_inverted(
    query_text=query_clean, 
    k=k_results, 
    vectorizer=vectorizer_no_sw, 
    tfidf_matrix=tfidf_matrix_no_sw, 
    df_documents=df_no_sw, 
    inverted_index=inverted_index_no_sw, 
    doc_id_to_idx=doc_id_to_idx,
    model_name="KHÔNG Stopwords"
)


**Truy vấn [CÓ Stopwords]:** 'cướp tiệm vàng ở huế'
-> Bộ lọc tìm thấy: 14766 văn bản ứng viên.
--- Top 5 Tài liệu tương đồng nhất ---

Rank 1 (Score: 0.6255):
  ID: 217072 | Title: Vụ cướp tiệm vàng tại chợ Đông Ba: Công an yêu cầu người dân trả lại vàng đã nhặt
  Content: vụ cướp tiệm vàng tại chợ đông_ba_công an yêu_cầu người_dân trả lại vàng đã nhặt sau khi bắt được nghi phạm trong vụ nổ_súng cướp tiệm vàng tại chợ đô...

Rank 2 (Score: 0.6235):
  ID: 217762 | Title: Công an đã bắt kẻ dùng súng cướp tiệm vàng ở Huế
  Content: công_an đã bắt kẻ dùng súng cướp tiệm vàng ở huế chiều date0731 lãnh_đạo công_an tỉnh thừa_thiên_huế cho biết hiện cơ_quan công_an đã bắt được đối_tượ...

Rank 3 (Score: 0.6107):
  ID: 217114 | Title: Nổ súng cướp tiệm vàng ở Huế: Công an đề nghị những người "hôi vàng" trả lại tài sản
  Content: nổ_súng cướp tiệm vàng ở huế_công an đề_nghị những người hôi vàng trả lại tài_sản liên_quan đến vụ nổ_súng cướp tiệm vàng ở huế chiều date0731 công_an...

Rank 4 (Sco

In [12]:
save_dir = 'models/'
os.makedirs(save_dir, exist_ok=True)

# Lưu mô hình CÓ Stopwords
with open(os.path.join(save_dir, 'tfidf_vectorizer_with_sw.pkl'), 'wb') as f:
    pickle.dump(vectorizer_with_sw, f)
save_npz(os.path.join(save_dir, 'tfidf_matrix_with_sw.npz'), tfidf_matrix_with_sw)

# Lưu mô hình KHÔNG Stopwords
with open(os.path.join(save_dir, 'tfidf_vectorizer_no_sw.pkl'), 'wb') as f:
    pickle.dump(vectorizer_no_sw, f)
save_npz(os.path.join(save_dir, 'tfidf_matrix_no_sw.npz'), tfidf_matrix_no_sw)

print(f"Đã lưu thành công toàn bộ 2 mô hình TF-IDF và Ma trận vào thư mục: {save_dir}")

Đã lưu thành công toàn bộ 2 mô hình TF-IDF và Ma trận vào thư mục: models/
